# 03 - Statistical Analysis

[Back to the EDA notebook](02_eda.ipynb)

This notebook explains the key statistical ideas we use, then tests two hypotheses that came out of the exploration.

## Setup: reusable functions and loading the data
We define the functions used in this notebook, then load the cleaned data and pick out total monthly receipts.

In [1]:

# Bring in pandas, our tool for working with tables
import pandas as pd


def load_clean_data(file_path="../data/processed/hmrc_clean.csv"):
    """Load the cleaned HMRC data with Month as the row label."""
    # Read the file, turning Month back into real dates
    data = pd.read_csv(file_path, parse_dates=["Month"])

    # Use Month as the row label and hand the table back
    return data.set_index("Month")


def describe_series(series):
    """Return the mean, median and standard deviation of a series."""
    # These three numbers summarise the middle and the spread of the data
    return pd.Series({"Mean": series.mean(),
                      "Median": series.median(),
                      "Standard deviation": series.std()}).round(0)


def probability_above(series, threshold):
    """Give the percentage of months where the value is above a threshold."""
    # (series > threshold) is True or False for each month, and the average of those is the share
    return round((series > threshold).mean() * 100, 1)


# Use our function to load the data
df = load_clean_data()

# Pull out total receipts, the series we'll explain the ideas with
total = df["Total HMRC Receipts"]

# Check the size, which should be (113, 44)
print(df.shape)

(113, 44)




**Action:** Defined three reusable functions (`load_clean_data`, `describe_series` and `probability_above`), then loaded the cleaned HMRC data and picked out total monthly receipts.

**Result:** 113 rows and 44 columns, with `Month` as the row label.

## Key concepts

### 1. Mean, median and standard deviation
- **Mean:** add up all the months and divide by how many there are. It is the "average", but a few very big months can pull it upwards.
- **Median:** line the months up from smallest to biggest and take the one in the middle. It isn't pulled around by extreme months.
- **Standard deviation:** a measure of how spread out the months are. A small number means months are all similar, and a large number means they swing a lot.

Here we use them on total monthly HMRC receipts, in £ million.

In [2]:

# Use our function to get the mean, median and standard deviation of total receipts
describe_series(total)

Mean                  61517.0
Median                59566.0
Standard deviation    16666.0
dtype: float64



**Action:** Calculated the mean, median and standard deviation of total monthly receipts.

**Result:** The mean is about £61.5bn and the median about £59.6bn. The standard deviation is about £16.7bn.

**Note:** The mean sits a little above the median, because a few very large months (the Januaries) pull the average up. The large standard deviation tells us receipts swing a lot from month to month, which matches the seasonal pattern we found in the EDA.

Uses function: `describe_series` (Cell 1)

### 2. Probability
Probability is the chance that something happens, from 0% (never) to 100% (always). With historical data we can work it out simply by counting: out of all the months we have, in what share did the thing happen?

For example: if 4 months out of 113 had receipts above £100bn, the chance of a month above £100bn is 4 out of 113, or about 3.5%.

In [3]:
# What share of months had total receipts above £100bn? (values are in £ million)
print("Above £100bn:", probability_above(total, 100000), "%")

# And above £80bn?
print("Above £80bn:", probability_above(total, 80000), "%")

# Which months were above £100bn?
print(total[total > 100000].index.strftime("%b %Y").tolist())

Above £100bn: 3.5 %
Above £80bn: 10.6 %
['Jan 2023', 'Jan 2024', 'Jan 2025', 'Jan 2026']


**Action:** Counted what share of the 113 months had total receipts above £100bn and above £80bn, and listed the months above £100bn.

**Result:** 3.5% of months (4 out of 113) were above £100bn, and 10.6% were above £80bn. The four months above £100bn were January 2023, 2024, 2025 and 2026.

**Note:** Every month above £100bn is a January, which is a strong clue that January really is different. We'll test that formally in the next section.

Uses function: `probability_above` (Cell 1)

### 3. Hypothesis testing
A hypothesis test asks: "Is what I see in the data real, or could it just be luck?"

- **Null hypothesis (H0):** nothing special is going on. For example, January receipts are no different from other months.
- **Alternative hypothesis (H1):** something is going on. For example, January receipts are higher than other months.
- **p-value:** the chance of seeing a difference this big if the null hypothesis were true. A very small p-value means luck is an unlikely explanation.
- **Significance level (alpha):** the cut-off we choose before testing, usually 0.05 (5%). If the p-value is below it, we reject the null hypothesis. If not, we say there isn't enough evidence.

**t-test:** compares the averages of two groups, for example January months against all other months, and gives a p-value.

### 4. Correlation
Correlation (r) measures how closely two things move together, from -1 to +1. A value near +1 means they rise and fall together, near 0 means no link, and near -1 means one rises while the other falls. The test also gives a p-value, telling us whether the link is likely to be real.

**Important:** correlation shows that two things move together, not that one causes the other.

### What comes next
We will test two hypotheses that came out of the EDA:
1. January receipts are higher than other months (t-test).
2. Income Tax and National Insurance receipts are positively correlated (correlation test).